In [39]:
import numpy as np
import pandas as pd
from scipy.stats import mode
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from ta import add_all_ta_features
import ta
from advanced_ta import LorentzianClassification
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

In [40]:

prediction = {
    'NEUTRAL': 0,
    'BUY': 1,
    'SELL': 2
}

In [41]:

eurusd = pd.read_csv("common/MachineLearningModel/output/fifteen_mins/EURUSD_15_Min.csv")
eurjpy = pd.read_csv('common/MachineLearningModel/output/fifteen_mins/EURJPY_15_Min.csv')
eurcad = pd.read_csv('common/MachineLearningModel/output/fifteen_mins/EURCAD_15_Min.csv')
euraud = pd.read_csv('common/MachineLearningModel/output/fifteen_mins/EURAUD_15_Min.csv')
eurgbp = pd.read_csv('common/MachineLearningModel/output/fifteen_mins/EURGBP_15_Min.csv')

In [42]:
from ta.trend import macd,cci,adx,macd_signal
from ta.momentum import rsi,stochrsi_d,stochrsi_k,stochrsi
def calculate(pd: pd.DataFrame):
    pdrsi = rsi(pd['close'],14)
    # rsi.dropna(axis=0,inplace=True)
    pdcci = cci(pd['high'],pd['low'],pd['close'],14)
    # cci.dropna(axis=0,inplace=True)
    pdadx = adx(pd['high'],pd['low'],pd['close'])
    # adx.dropna(axis=0,inplace=True)
    pdmacd = macd(pd['close'])
    # macd.dropna(axis=0,inplace=True)
    pdmacd_signal = macd_signal(pd['close'])
    # macd_signal.dropna(axis=0,inplace=True)
    pdstochrsi_d = stochrsi_d(pd['close'])
    pdstochrsi_k = stochrsi_k(pd['close'])
    pdstochrsi = stochrsi(pd['close'])
    # stochrsi.dropna(axis=0,inplace=True)
    pd2 = pd.iloc[:,1:7].copy(deep=True) # iloc[row,column]
    pd2['rsi'] = pdrsi
    pd2['cci'] = pdcci
    pd2['adx'] = pdadx
    pd2['macd'] = pdmacd
    pd2['macd_signal'] = pdmacd_signal
    pd2['stochrsi_d'] = pdstochrsi_d
    pd2['stochrsi_k'] = pdstochrsi_k
    pd2['stochrsi'] = pdstochrsi
    return pd2


In [43]:
pd1 = calculate(eurusd)
pd2 = calculate(eurjpy)
pd3 = calculate(eurcad)
pd4 = calculate(euraud)
pd5 = calculate(eurgbp)

In [44]:
data = pd.concat([pd1,pd2,pd3,pd4,pd5])
data.dropna(axis=0,inplace=True)
print(data.columns)
print(data.count())

Index(['symbol', 'open', 'high', 'low', 'close', 'volume', 'rsi', 'cci', 'adx',
       'macd', 'macd_signal', 'stochrsi_d', 'stochrsi_k', 'stochrsi'],
      dtype='object')
symbol         29814
open           29814
high           29814
low            29814
close          29814
volume         29814
rsi            29814
cci            29814
adx            29814
macd           29814
macd_signal    29814
stochrsi_d     29814
stochrsi_k     29814
stochrsi       29814
dtype: int64


In [45]:
data['RSI_1'] = np.where(data['rsi'] < 40, 1, np.where(data['rsi'] > 60, 2, 0))
# data['MACD_1'] = np.where(data['macd'] < data['macd_signal'], 2, np.where(data['macd'] > data['macd_signal'], 1, 0))
# data['CCI_1'] = np.where(data['cci'] < -80, 1, np.where(data['cci'] > 80, 2, 0))
data['ADX_1'] = np.where(data['adx'] > 25, 1, 0)
# conditions_3 = (data['stochrsi'] > 0.75) & (data['stochrsi_k'] < data['stochrsi_d'])
# conditions_4 = (data['stochrsi'] < 0.25) & (data['stochrsi_k'] > data['stochrsi_d'])
# data['STOCH.RSI'] = np.where(conditions_3, 2, np.where(conditions_4, 1, 0))
# conditions_1 = (data['ADX_1'] == 1)  #(data['ADX_1'] == 1) & (data['MACD_1'] == 1) #(data['STOCH.RSI'] == 1) # & (data['CCI_1'] == 1) & (data['MACD_1'] == 1) (data['ADX_1'] == 1) & (data['RSI_1'] == 1) & 
conditions_1 = (data['RSI_1'] == 1)  #(data['ADX_1'] == 1) & (data['MACD_1'] == 1) #(data['STOCH.RSI'] == 1) # & (data['CCI_1'] == 1) & (data['MACD_1'] == 1) (data['ADX_1'] == 1) & (data['RSI_1'] == 1) & 
# conditions_2 = (data['ADX_1'] == 1) #(data['ADX_1'] == 1) & (data['MACD_1'] == 2) #(data['STOCH.RSI'] == 2) # & (data['CCI_1'] == 2) & (data['MACD_1'] == 2) & (data['ADX_1'] == 1)  (data['RSI_1'] == 2) & 
conditions_2 = (data['RSI_1'] == 2) #(data['ADX_1'] == 1) & (data['MACD_1'] == 2) #(data['STOCH.RSI'] == 2) # & (data['CCI_1'] == 2) & (data['MACD_1'] == 2) & (data['ADX_1'] == 1)  (data['RSI_1'] == 2) & 
data['Prediction'] = np.where(conditions_1 & (data['open'] < data['close']), 1,
                              np.where(conditions_1 & (data['open'] > data['close']), 2, 0))


In [46]:
# data.drop(axis=1,labels=['rsi','cci','adx','macd','macd_signal','stochrsi','stochrsi_k','stochrsi_d'],inplace=True)
data.drop(axis=1,labels=['RSI_1'],inplace=True)
data.drop(axis=1,labels=['ADX_1'],inplace=True)

In [47]:
data.dropna(inplace=True)
data.reset_index(drop=True,inplace=True)
print(data.head())
print(data.shape)


      symbol     open     high      low    close  volume        rsi  \
0  FX:EURUSD  1.09691  1.09719  1.09679  1.09692  2601.0  22.635875   
1  FX:EURUSD  1.09692  1.09694  1.09613  1.09655  2215.0  21.530159   
2  FX:EURUSD  1.09655  1.09681  1.09580  1.09607  2717.0  20.154702   
3  FX:EURUSD  1.09607  1.09634  1.09535  1.09538  3207.0  18.340813   
4  FX:EURUSD  1.09538  1.09586  1.09532  1.09578  3561.0  22.684860   

          cci        adx      macd  macd_signal  stochrsi_d  stochrsi_k  \
0 -125.095024  44.853912 -0.001531    -0.001047    0.000359    0.001077   
1 -114.775738  46.664536 -0.001634    -0.001165    0.000718    0.001077   
2 -106.059162  48.446073 -0.001734    -0.001279    0.001077    0.001077   
3 -106.771273  50.231559 -0.001848    -0.001392    0.000718    0.000000   
4  -96.416773  51.898280 -0.001885    -0.001491    0.024446    0.072260   

   stochrsi  Prediction  
0  0.003232           1  
1  0.000000           2  
2  0.000000           2  
3  0.000000       

In [48]:


le = LabelEncoder()
le.fit_transform(data['Prediction'])
print(le.classes_)


[0 1 2]


In [49]:
print(data['Prediction'].value_counts())
X = data.iloc[:,6:-1]
y = data.iloc[:, -1]
print(data.columns)
print(X.head())
print(X.count())
# print(y.head())
X_train, X_test, y_train, y_test =train_test_split(
  X, y, test_size = 0.2, random_state = 24)



Prediction
0    24735
2     3525
1     1554
Name: count, dtype: int64
Index(['symbol', 'open', 'high', 'low', 'close', 'volume', 'rsi', 'cci', 'adx',
       'macd', 'macd_signal', 'stochrsi_d', 'stochrsi_k', 'stochrsi',
       'Prediction'],
      dtype='object')
         rsi         cci        adx      macd  macd_signal  stochrsi_d  \
0  22.635875 -125.095024  44.853912 -0.001531    -0.001047    0.000359   
1  21.530159 -114.775738  46.664536 -0.001634    -0.001165    0.000718   
2  20.154702 -106.059162  48.446073 -0.001734    -0.001279    0.001077   
3  18.340813 -106.771273  50.231559 -0.001848    -0.001392    0.000718   
4  22.684860  -96.416773  51.898280 -0.001885    -0.001491    0.024446   

   stochrsi_k  stochrsi  
0    0.001077  0.003232  
1    0.001077  0.000000  
2    0.001077  0.000000  
3    0.000000  0.000000  
4    0.072260  0.216780  
rsi            29814
cci            29814
adx            29814
macd           29814
macd_signal    29814
stochrsi_d     29814
stochrsi_

In [50]:
# Initialize XGBoost classifier
xgb_model = XGBClassifier(booster="gbtree",max_depth=8,min_child_weight = 2)
# Train the model
xgb_model.fit(X_train, y_train)

# Make predictions on the test set
preds = xgb_model.predict(X_test)

# Evaluate the model
print(f"Accuracy on train data by XGBoost Classifier\
: {accuracy_score(y_train, xgb_model.predict(X_train))*100}")
 
print(f"Accuracy on test data by XGBoost Classifier\
: {accuracy_score(y_test, preds)*100}")


Accuracy on train data by XGBoost Classifier: 100.0
Accuracy on test data by XGBoost Classifier: 98.08821063223209


In [51]:
final_xgb_model = XGBClassifier()
final_xgb_model.fit(X, y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, objective='multi:softprob', ...)

In [52]:
import pickle
xgb_final_model = pickle.dump(final_xgb_model, open('xgbclassifier_15.sav','wb'))

In [53]:
from TradingDataGenerate import main
s = main.TvDatafeed('mageshragav1@gmail.com','Magesh1@')


ERROR:TradingDataGenerate.main:error while signin


In [55]:
symbols = 'EURUSD'
response_data = s.get_hist(symbol=symbols,exchange='FX',interval=main.Interval.in_15_minute,n_bars=150,extended_session=False)
pd3 = calculate(response_data)
# pd3['ADX_1'] = np.where(pd3['adx'] > 25, 1, 0)
pd3.dropna(inplace=True)
pd3.reset_index()
print(pd3.iloc[-1])
data_1 = pd3.iloc[-1,5:].to_dict()
data_1 = pd.DataFrame({key: [value] for key, value in data_1.items()})
print(data_1)
output = final_xgb_model.predict(pd.DataFrame(data_1))
print(output)

open             1.080130
high             1.080230
low              1.080010
close            1.080080
volume         717.000000
rsi             50.461560
cci            -22.879859
adx             26.889725
macd             0.000099
macd_signal      0.000064
stochrsi_d       0.460259
stochrsi_k       0.395930
stochrsi         0.369970
Name: 2024-03-28 11:30:00, dtype: float64
        rsi        cci        adx      macd  macd_signal  stochrsi_d  \
0  50.46156 -22.879859  26.889725  0.000099     0.000064    0.460259   

   stochrsi_k  stochrsi  
0     0.39593   0.36997  
[0]
